In [1]:
# Load packages
import h5py
import numpy as np
import torch
import torch.nn.functional as F
from tqdm import tqdm
from torch.utils.data import random_split
from torch_geometric.data import Data, Dataset
from torch_geometric.loader import DataLoader
from torch_geometric.nn import radius_graph
import torch.nn as nn
from torch_geometric.utils import to_dense_batch, to_dense_adj
from equiformer_pytorch import Equiformer
import os
import torch
from torch.optim import Adam
from torch.optim.lr_scheduler import StepLR
from torch.utils.tensorboard import SummaryWriter
import torch
import torch.nn.functional as F
from tqdm import tqdm

In [2]:
# Preparing the ANI-1x dataset
# ANI1-1x elements 
element_to_idx = {"H": 0, "C":1, "N":2, "O":3}
n_elements = len(element_to_idx)

# Unit conversion
Hartree_to_eV = 27.211386245988
default_cutoff= 5.0

# Keys inside ANI1x
energy_key = "wb97x_tz.energy"
forces_key = "wb97x_td.forces"

# OHE
def one_hot_species(species_bytes):
    indices = [element_to_idx[s.decode()] for s in species_bytes]
    idx_t = torch.tensor(indices, dtype= torch.long)
    return F.one_hot(idx_t, num_classes= n_elements).float()

In [3]:
# graph edges from 3D positions
def radius_edge_index(pos, cutoff= default_cutoff):
    edge_index = radius_graph(pos, r= cutoff, loop=False)
    row, col = edge_index
    edge_attr = (pos[row] - pos[col]).norm(dim=1, keepdim=True)
    return  edge_attr, edge_index

In [4]:
# One PYG-data object
def pyg_data(species_bytes, coords, energy, forces=None, cutoff= default_cutoff, convert_units=True):
    x = one_hot_species(species_bytes)
    pos = torch.tensor(coords, dtype = torch.float32)
    energy_t = torch.tensor([energy], dtype= torch.float32)
    if convert_units:
        energy_t = energy_t * Hartree_to_eV

    edge_attr, edge_index = radius_edge_index(pos, cutoff)
    data = Data(x=x, pos=pos, y=energy_t, edge_index=edge_index, edge_attr=edge_attr)
    if forces is not None:
        forces_t = torch.tensor(forces, dtype=torch.float32)
        if convert_units:
            forces_t = forces_t * Hartree_to_eV
        data.forces = forces.t

    return data

In [5]:
# dataset class
class ANI1xPyGDataset(Dataset):
    def __init__(self, h5_path, cutoff= default_cutoff, convert_units= True, load_forces = True, max_samples=None, energy_key= energy_key, forces_key= forces_key):
        super().__init__()
        self.data_list=[]
        self._load(h5_path= h5_path, cutoff=cutoff, convert_units= convert_units, load_forces= load_forces, max_samples= max_samples, energy_key= energy_key, forces_key= forces_key)
    def _load(self, h5_path, cutoff, convert_units, load_forces, max_samples, energy_key, forces_key):
        print(f"loading ANI-1x from:{h5_path}")
        count = 0
        with h5py.File(h5_path, "r") as f:
            for mol_key in tqdm(f.keys(), desc= "Molecules"):
                grp = f[mol_key]
                # species
                if "species" in grp:
                    species_bytes = list(grp["species"][()])
                elif "species" in f:
                    species_bytes= list(f["species"][()])
                elif "atomic_numbers" in grp:
                    z = grp["atomic_numbers"][()]
                    z_to_sym= {1: "H", 6:"C", 7:"N", 8:"O"}
                    species_bytes = [z_to_sym[int(zi)].encode()for zi in z]
                else:
                    raise KeyError(
                        f"No species/atomic_numbers found for {mol_key}."
                        f"Available keys: {list(grp.keys())}"
                    )

                # coordinates
                if "coordinates" in grp:
                    coords_all = grp["coordinates"][()]
                elif "xyz" in grp:
                    coords_all = grp["xyz"][()]
                else:
                    raise KeyError(
                        f"No coordinates/xyz found for {mol_key}."
                        f"Available keys: {list(grp.keys())}"
                    )
                # energy 
                if "energies" in grp:
                    energies_all = list(grp["energies"][()])
                elif "energy" in f:
                    energies_all= list(f["enregy"][()])
                elif energy_key in grp:
                    energies_all = grp[energy_key][()]
                else:
                    raise KeyError(
                        f"No energies found for {mol_key}."
                        f"Tried: 'energies', 'ennergy', '{energy_key}.'"
                        f"Available keys: {list(grp.keys())}"
                    )

                # forces 
                forces_all = None
                if load_forces:
                    if "forces" in grp:
                        forces_all = grp["forces"][()]
                    elif "forces" in grp:
                        forces_all = grp["forces"][()]
                    elif forces_key in grp:
                        forces_all = grp[forces_key][()]
                    else:
                        forces_all= None
                    n_conf= coords_all.shape[0]

                    if len(energies_all) != n_conf:
                        raise ValueError(
                            f"{mol_key}: energies length {len(energies_all)} != n_conf {n_conf}."
                        )
                    for i in range (n_conf):
                        forces_i = forces_all[i] if forces_all is not None else None
                        data = pyg_data(
                            species_bytes= species_bytes, 
                            coords= coords_all[i], 
                            energy=float(energies_all[i]), 
                            forces= forces_i, 
                            cutoff= cutoff, 
                            convert_units= convert_units
                        )
                        self.data_list.append(data)
                        
                        count+=1

                        if max_samples is not None and count >= max_samples:
                            print(f"Reached max_samples={max_samples}, stopping early.")
                            return

        print(f"Done --- loaded{len(self.data_list):,} conformation")

    def len(self):
        return len(self.data_list)

    def get(self, idx):
        return self.data_list[idx]

In [8]:
# dataloader
def dataloader(h5_path, batch_size = 8, 
               cutoff= default_cutoff, 
               convert_units= True, 
               load_forces = True, 
               max_samples= None, 
               train_frac= 0.90, 
               val_frac= 0.05, 
               num_workers= 0, 
               seed =42, 
               energy_key = energy_key, 
               forces_key= forces_key
              ):
    dataset=ANI1xPyGDataset(h5_path=h5_path, 
                           cutoff= cutoff,
                           convert_units= convert_units,
                           load_forces= load_forces,
                           max_samples= max_samples,
                           energy_key=energy_key,
                           forces_key=forces_key
                           )
    n_total = len(dataset)
    n_train = int(n_total * train_frac)
    n_val = int(n_total *val_frac)
    n_test = n_total - n_train - n_val

    print(f"Split --> train: {n_train:,} | val : {n_val:,} | test: {n_test:,}")

    generator = torch.Generator().manual_seed(seed)
    train_set, val_set, test_set =random_split(
        dataset, [n_train, n_val, n_test], generator = generator
    )
    shared = dict(num_workers=num_workers, pin_memory= torch.cuda.is_available())
    train_loader = DataLoader(train_set, batch_size= batch_size, shuffle= True, **shared)
    val_loader = DataLoader(val_set, batch_size= batch_size, shuffle= False, **shared)
    test_loader= DataLoader(test_set, batch_size= batch_size, shuffle= False, **shared)


    return train_loader, val_loader, test_loader

In [10]:
# Calling data from Path
h5_path = "/home/ismail/db6/drug_discovery_ai/data/ANI1x/ani1x-release.h5"

# Load data - Spliting
train_loader, val_loader, test_loader = dataloader(
    h5_path=h5_path,
    batch_size=8,
    cutoff=5.0,
    load_forces=True,
    max_samples=4000000,  
    num_workers=0
)

loading ANI-1x from:/home/ismail/db6/drug_discovery_ai/data/ANI1x/ani1x-release.h5


Molecules:  70%|███████   | 2187/3114 [23:07<09:48,  1.58it/s]  

Reached max_samples=4000000, stopping early.
Split --> train: 3,600,000 | val : 200,000 | test: 200,000


In [11]:
def compute_zscore_stats(loader):

    ys = []

    for batch in loader:
        ys.append(batch.y)

    ys = torch.cat(ys, dim=0)

    mean = ys.mean()
    std  = ys.std()

    return mean, std
    
def zscore_normalize(y, mean, std):
    return(y - mean)/std
    
def zscore_denormalize(y_norm, mean, std):
    return y_norm * std + mean

In [12]:
y_mean, y_std = compute_zscore_stats(train_loader)

print("Training energy mean:", y_mean.item())
print("Training energy std :", y_std.item())

Training energy mean: nan
Training energy std : nan


In [13]:
def compute_zscore_stats(loader, max_batches=None):

    ys = []
    for i, batch in enumerate(loader):

        ys.append(batch.y)

        if max_batches and i >= max_batches:
            break

    ys = torch.cat(ys)

    return ys.mean(), ys.std()

In [14]:
y_mean, y_std = compute_zscore_stats(train_loader, max_batches=500)

In [15]:
# Equiformer
class EquiformerANI1x(nn.Module):
    def __init__(self, n_token=n_elements, n_out=1, hidden_dim=128, depth=4):
        super().__init__()

        self.hidden_dim = hidden_dim

        # Atom feature embedding 
        self.embedding = nn.Linear(n_token, hidden_dim)

        # Equiformer core
        self.model = Equiformer(
            dim=hidden_dim,
            dim_in=hidden_dim,
            input_degrees=1,
            num_degrees=2,

            heads=4,
            dim_head=hidden_dim // 4,
            depth=depth,

            attend_sparse_neighbors=True,
            num_neighbors=0,
            num_adj_degrees_embed=2,
            max_sparse_neighbors=16,
            valid_radius=default_cutoff,

            reduce_dim_out=False,
            attend_self=True,
            l2_dist_attention=False
        )

        # Regression head: one energy value
        self.linear = nn.Linear(hidden_dim, n_out)

    def forward(self, data):
        x = self.encode(data)              
        x = self.linear(x)                
        return x.squeeze(-1)               

    def encode(self, data):
        x, coords, batch = data.x, data.pos, data.batch

        # Embed node features
        x = self.embedding(x)              
        # Dense batching
        x, mask = to_dense_batch(x, batch)        
        coords, _ = to_dense_batch(coords, batch) 

        # Dense adjacency from PyG edge_index
        adj_mat = to_dense_adj(data.edge_index, batch=batch).bool() 
        
        # Equiformer forward
        out = self.model(x, coords, mask=mask, adj_mat=adj_mat)

        # Extract invariant degree-0 features
        if hasattr(out, "type0"):
            x = out.type0
        elif isinstance(out, dict):
            x = out["type0"] if "type0" in out else next(iter(out.values()))
        elif isinstance(out, (list, tuple)):
            x = out[0]
        else:
            x = out

        # Masked mean pooling over atoms
        mask_f = mask[:, :x.size(1)].float()
        x = (x * mask_f.unsqueeze(-1)).sum(dim=1) / mask_f.sum(dim=1, keepdim=True).clamp_min(1.0)

        return x  

In [16]:
model = EquiformerANI1x(n_token=n_elements, n_out=1, hidden_dim=128, depth=4)
print(model)

EquiformerANI1x(
  (embedding): Linear(in_features=4, out_features=128, bias=True)
  (model): Equiformer(
    (tp_in): DTP(
      (to_xi): Linear(
        (weights): ParameterList(  (0): Parameter containing: [torch.float32 of size 128x128])
      )
      (to_xj): Linear(
        (weights): ParameterList(  (0): Parameter containing: [torch.float32 of size 128x128])
      )
      (kernel_unary): ModuleDict(
        ((0,0)): Radial(
          (rp): Sequential(
            (0): Linear(in_features=1, out_features=64, bias=True)
            (1): SiLU()
            (2): LayerNorm()
            (3): Linear(in_features=64, out_features=64, bias=True)
            (4): SiLU()
            (5): LayerNorm()
            (6): Linear(in_features=64, out_features=16384, bias=True)
            (7): Rearrange('... (lo li) -> ... lo li', li=128, lo=128)
          )
        )
        ((0,1)): Radial(
          (rp): Sequential(
            (0): Linear(in_features=1, out_features=64, bias=True)
            

In [17]:
def count_parameters(model):
    return sum(p.numel() for p in model.parameters() if p.requires_grad)

In [18]:
print("Trainable parameters:", count_parameters(model))

Trainable parameters: 18588833


In [19]:
# Z-score helpers
def zscore_normalize(y, mean, std):
    return (y - mean) / std

def zscore_denormalize(y_norm, mean, std):
    return y_norm * std + mean

# Train epoch
def train_epoch(model, loader, optimizer, device, y_mean, y_std, accum_steps=16):
    """
    Perform one epoch of training.

    Args:
        model: PyTorch model
        loader: training DataLoader
        optimizer: optimizer (Adam, etc.)
        device: cpu or cuda
        y_mean: training target mean
        y_std: training target std
        accum_steps: gradient accumulation steps

    Returns:
        Average training loss
    """

    model.train()

    scaler = torch.cuda.amp.GradScaler()
    optimizer.zero_grad()

    total_loss = 0.0

    for step, batch in enumerate(tqdm(loader, desc="Training", ncols=80)):

        batch = batch.to(device)

        # original targets (eV)
        true_ev = batch.y.view(-1)

        # normalized targets
        true_norm = zscore_normalize(true_ev, y_mean, y_std)

        with torch.cuda.amp.autocast():

            pred_norm = model(batch).view(-1)

            # MSE loss in normalized space
            loss = F.mse_loss(pred_norm, true_norm)

            # gradient accumulation scaling
            loss = loss / accum_steps

        scaler.scale(loss).backward()

        if (step + 1) % accum_steps == 0 or (step + 1) == len(loader):

            scaler.step(optimizer)
            scaler.update()

            optimizer.zero_grad()

        total_loss += loss.item() * accum_steps

    return total_loss / len(loader)

# Evaluate model


@torch.no_grad()
def evaluate(model, loader, device, y_mean, y_std):
    """
    Evaluate model MAE in eV.

    Args:
        model: PyTorch model
        loader: validation or test DataLoader
        device: cpu or cuda
        y_mean: normalization mean
        y_std: normalization std

    Returns:
        MAE in eV
    """

    model.eval()

    preds = []
    targets = []

    for batch in tqdm(loader, desc="Eval", leave=False):

        batch = batch.to(device)

        # normalized predictions
        pred_norm = model(batch).view(-1)

        # convert back to eV
        pred_ev = zscore_denormalize(pred_norm, y_mean, y_std)

        preds.append(pred_ev.detach().cpu())
        targets.append(batch.y.view(-1).detach().cpu())

    preds = torch.cat(preds, dim=0)
    targets = torch.cat(targets, dim=0)

    mae = torch.mean(torch.abs(preds - targets)).item()

    return mae

In [20]:
# Training Loop 
def main():
    # Hyperparameters
    epochs = 100
    lr = 3e-4
    lr_decay_step = 50
    lr_decay_factor = 0.5
    weight_decay = 1e-4
    accum_steps = 4

    save_dir = "checkpoints_Equiformer_ANI1x"
    log_dir = "logs_Equiformer_ANI1x"

    # Create directories for checkpt and logs
    os.makedirs(save_dir, exist_ok=True)
    os.makedirs(log_dir, exist_ok=True)

    writer = SummaryWriter(log_dir=log_dir)
    # Device
    device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
    print("Device:", device)
    if device.type == "cuda":
        print("GPU:", torch.cuda.get_device_name(0))

    # Model
    model = EquiformerANI1x(
        n_token=n_elements,
        n_out=1,
        hidden_dim=128,
        depth=4
    ).to(device)

    # Optimizer and Scheduler
    optimizer = Adam(model.parameters(), lr=lr, weight_decay=weight_decay)
    scheduler = StepLR(optimizer, step_size=lr_decay_step, gamma=lr_decay_factor)

    # Move normalization stats to device
    y_mean_dev = y_mean.to(device)
    y_std_dev = y_std.to(device)

    # Track best validation/test performance
    best_val_mae = float("inf")
    best_test_mae = float("inf")

    # Print model-info
    n_params = sum(p.numel() for p in model.parameters() if p.requires_grad)
    print("#params:", n_params)
    print("Training for single target: ANI-1x energy")

    # Training Loop
    for epoch in range(1, epochs + 1):
        print(f"\n=== Epoch {epoch} ===")

        # Train for one epoch (loss in normalized space)
        train_loss = train_epoch(
            model=model,
            loader=train_loader,
            optimizer=optimizer,
            device=device,
            y_mean=y_mean_dev,
            y_std=y_std_dev,
            accum_steps=accum_steps
        )

        # Evaluate MAE in eV
        val_mae = evaluate(
            model=model,
            loader=val_loader,
            device=device,
            y_mean=y_mean_dev,
            y_std=y_std_dev
        )

        test_mae = evaluate(
            model=model,
            loader=test_loader,
            device=device,
            y_mean=y_mean_dev,
            y_std=y_std_dev
        )

        current_lr = optimizer.param_groups[0]["lr"]

        # Print metrics
        print(f"Train loss (normalized MSE): {train_loss:.6f}")
        print(f"Val MAE (eV):              {val_mae:.6f}")
        print(f"Test MAE (eV):             {test_mae:.6f}")
        print(f"LR:                        {current_lr:.2e}")

        # TensorBoard logs
        writer.add_scalar("loss/train_mse_norm", train_loss, epoch)
        writer.add_scalar("mae/val_ev", val_mae, epoch)
        writer.add_scalar("mae/test_ev", test_mae, epoch)
        writer.add_scalar("lr", current_lr, epoch)

        # Save best model checkpoint 
        if val_mae < best_val_mae:
            best_val_mae = val_mae
            best_test_mae = test_mae

            save_path = os.path.join(save_dir, "best_model.pt")
            torch.save(
                {
                    "epoch": epoch,
                    "model_state_dict": model.state_dict(),
                    "optimizer_state_dict": optimizer.state_dict(),
                    "best_val_mae": best_val_mae,
                    "best_test_mae": best_test_mae,
                    "y_mean": y_mean.detach().cpu(),
                    "y_std": y_std.detach().cpu(),
                },
                save_path
            )

            print(f"Saved best model (Val MAE improved to {best_val_mae:.6f} eV)")

        scheduler.step()

    writer.close()

    print("\nFinished training.")
    print(f"Best validation MAE (eV): {best_val_mae:.6f}")
    print(f"Test MAE at best val (eV): {best_test_mae:.6f}")

if __name__ == "__main__":
    main()

Device: cuda
GPU: Quadro RTX 8000
#params: 18588833
Training for single target: ANI-1x energy

=== Epoch 1 ===


Training: 100%|██████████████████████| 450000/450000 [16:08:22<00:00,  7.74it/s]
/home/ismail/dig_envi/lib/python3.9/site-packages/torch/optim/lr_scheduler.py:139: UserWarning: Detected call of `lr_scheduler.step()` before `optimizer.step()`. In PyTorch 1.1.0 and later, you should call them in the opposite order: `optimizer.step()` before `lr_scheduler.step()`.  Failure to do this will result in PyTorch skipping the first value of the learning rate schedule. See more details at https://pytorch.org/docs/stable/optim.html#how-to-adjust-learning-rate
  warnings.warn("Detected call of `lr_scheduler.step()` before `optimizer.step()`. "


Train loss (normalized MSE): nan
Val MAE (eV):              nan
Test MAE (eV):             nan
LR:                        3.00e-04

=== Epoch 2 ===


Training:  29%|█████▊              | 131306/450000 [4:35:51<11:09:31,  7.93it/s]


KeyboardInterrupt: 

In [21]:
batch = next(iter(train_loader))

print("batch.y shape:", batch.y.shape)
print("batch.y dtype:", batch.y.dtype)
print("any nan in y:", torch.isnan(batch.y).any().item())
print("any inf in y:", torch.isinf(batch.y).any().item())
print("y min:", batch.y.min().item())
print("y max:", batch.y.max().item())

batch.y shape: torch.Size([8])
batch.y dtype: torch.float32
any nan in y: True
any inf in y: False
y min: nan
y max: nan


In [22]:
print("y_mean:", y_mean)
print("y_std :", y_std)
print("any zero std:", (y_std == 0).any().item() if y_std.ndim > 0 else (y_std.item() == 0))
print("any nan std:", torch.isnan(y_std).any().item() if torch.is_tensor(y_std) else False)

y_mean: tensor(nan)
y_std : tensor(nan)
any zero std: False
any nan std: True


In [24]:
def inspect_nan_targets_subset(subset, n=20):
    found = 0
    for j in range(len(subset)):
        i = subset.indices[j]
        y = subset.dataset[i].y
        if torch.isnan(y).any() or torch.isinf(y).any():
            print(f"Subset item {j} -> original index {i} has invalid y: {y}")
            found += 1
            if found >= n:
                break
    print("Done.")

inspect_nan_targets_subset(train_loader.dataset, n=10)

Subset item 3 -> original index 3847005 has invalid y: tensor([nan])
Subset item 8 -> original index 2096294 has invalid y: tensor([nan])
Subset item 47 -> original index 3757044 has invalid y: tensor([nan])
Subset item 62 -> original index 406948 has invalid y: tensor([nan])
Subset item 66 -> original index 3499577 has invalid y: tensor([nan])
Subset item 82 -> original index 2812260 has invalid y: tensor([nan])
Subset item 98 -> original index 1963363 has invalid y: tensor([nan])
Subset item 139 -> original index 2558741 has invalid y: tensor([nan])
Subset item 149 -> original index 3304243 has invalid y: tensor([nan])
Subset item 161 -> original index 3034873 has invalid y: tensor([nan])
Done.
